# Chapter 29 — Machine Learning and Text for Economists

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch29_ml_and_text.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 29.1 Prediction is not causation

## 29.2 The data: women's labour force participation in PLFS 2023–24

In [ ]:
import html
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from indiadata import read_plfs, labour_status, bis, mospi_cpi

KEY = ["qtr", "mfsu", "seg", "sss", "ssu"]          # identifies a household
person = read_plfs("perv1", columns=["seg", "sss", "ssu", "rel", "sex", "age",
                                     "marst", "gedu_lvl", "pas", "sas",
                                     "acws", "ern_reg", "aind_cws"])
house = read_plfs("hhv1", columns=["seg", "sss", "ssu", "hh_size", "sg",
                                   "relg", "hce_tot"])
print("household key unique:", not house.duplicated(KEY).any())

p = person.merge(house[KEY + ["hh_size", "sg", "relg", "hce_tot"]],
                 on=KEY, how="left", validate="many_to_one")
del person, house                      # free the memory straight away
print(f"{len(p):,} persons; without a household record: "
      f"{p['hh_size'].isna().sum()}")

In [ ]:
p["status"] = labour_status(p["pas"], p["sas"])
women = p[(p["sex"] == 2) & p["age"].between(15, 59)].copy()
women["in_lf"] = (women["status"] != "out of LF").astype(int)
women["log_mpce"] = np.log(women["hce_tot"] / women["hh_size"])

NUM = ["age", "hh_size", "log_mpce"]                          # numbers
CAT = ["gedu_lvl", "marst", "rel", "sec", "st", "sg", "relg"]  # codes
lfpr = np.average(women["in_lf"], weights=women["weight"])
print(f"{len(women):,} women aged 15-59; weighted LFPR {lfpr:.1%}")
women[NUM + CAT + ["in_lf"]].head(3)

In [ ]:
sal = p[(p["acws"] == 31) & (p["ern_reg"] > 0) & p["age"].between(15, 64)]
sal = sal[["mfsu", "weight", "sex", "age", "marst", "gedu_lvl", "sec", "st",
           "aind_cws", "ern_reg"]].astype("float64")
del p
print(f"{len(sal):,} regular salaried workers with earnings")

## 29.3 Train, test, and the survey design

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

X = women[NUM + CAT].astype("float64")
y = women["in_lf"].to_numpy()
fsu = women["mfsu"].to_numpy()
wt = women["weight"].to_numpy()

split = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=29)
tr, te = next(split.split(X, y, groups=fsu))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
w_tr = wt[tr] / wt[tr].mean()          # weights rescaled to average 1
w_te = wt[te]
print(f"train {len(tr):,} women in {len(set(fsu[tr])):,} FSUs; "
      f"test {len(te):,} in {len(set(fsu[te])):,} FSUs; "
      f"FSUs in both: {len(set(fsu[tr]) & set(fsu[te]))}")

### Two baselines and a logit

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def scores(prob, y=y_te, w=w_te):
    """Weighted test-set accuracy, AUC and log loss of predicted probabilities."""
    return pd.Series({"accuracy": accuracy_score(y, prob >= 0.5, sample_weight=w),
                      "AUC": roc_auc_score(y, prob, sample_weight=w),
                      "log loss": log_loss(y, prob, sample_weight=w)})


def logit(num, cat):
    prep = ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(drop="first",
                                                    handle_unknown="ignore"), cat)])
    return make_pipeline(prep, LogisticRegression(C=np.inf, max_iter=2000))


share_lf = np.average(y_tr, weights=w_tr)          # learnt on training data
results = {"Baseline: everyone out": scores(np.full(len(te), share_lf))}

small = logit(["age"], ["gedu_lvl", "marst", "sec"])
small.fit(X_tr, y_tr, logisticregression__sample_weight=w_tr)
results["Logit, 4 features"] = scores(small.predict_proba(X_te)[:, 1])

full = logit(NUM, CAT)
full.fit(X_tr, y_tr, logisticregression__sample_weight=w_tr)
p_logit = full.predict_proba(X_te)[:, 1]
results["Logit, all features"] = scores(p_logit)
pd.DataFrame(results).T.round(3)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_te, p_logit >= 0.5, sample_weight=w_te, normalize="all")
cm = pd.DataFrame(cm, index=["actually out", "actually in"],
                  columns=["predicted out", "predicted in"])
print((100 * cm).round(1))
print(f"Share of women in the labour force that the logit finds (recall): "
      f"{cm.loc['actually in', 'predicted in'] / cm.loc['actually in'].sum():.1%}")

## 29.4 Overfitting, and cross-validation

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [1, 2, 3, 4, 6, 8, 10, 12, 15, 20, 25]
curve = []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0)
    tree.fit(X_tr, y_tr, sample_weight=w_tr)
    curve.append({"depth": d, "leaves": tree.get_n_leaves(),
                  "train AUC": roc_auc_score(y_tr, tree.predict_proba(X_tr)[:, 1],
                                             sample_weight=w_tr),
                  "test AUC": roc_auc_score(y_te, tree.predict_proba(X_te)[:, 1],
                                            sample_weight=w_te)})
curve = pd.DataFrame(curve).set_index("depth")
curve.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(curve.index, curve["train AUC"], marker="o", color="#9db7d5",
        label="Training data (seen)")
ax.plot(curve.index, curve["test AUC"], marker="o", color="#1f4e79",
        label="Test data (unseen)")
best = curve["test AUC"].idxmax()
ax.axvline(best, color="#c55a11", ls=":", lw=1)
ax.annotate("overfitting: the tree memorises\nthe training sample",
            xy=(20, curve.loc[20, "test AUC"]), xytext=(13.5, 0.88),
            fontsize=9, color="#c55a11",
            arrowprops=dict(arrowstyle="->", color="#c55a11"))
ax.set_xlabel("Tree depth (number of questions asked)")
ax.set_ylabel("AUC")
ax.set_title("A deeper tree always fits better in sample, not out of sample",
             loc="left", fontsize=13)
ax.legend(loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Women aged 15-59, labour force participation (usual "
         "status). Source: PLFS 2023-24 unit-level data (NSO, MoSPI), weighted.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
cv = GroupKFold(n_splits=5)
cv_auc = {}
for d in [4, 6, 8, 10, 12]:
    fold_scores = []
    for fit_idx, val_idx in cv.split(X_tr, y_tr, groups=fsu[tr]):
        tree = DecisionTreeClassifier(max_depth=d, random_state=0)
        tree.fit(X_tr.iloc[fit_idx], y_tr[fit_idx], sample_weight=w_tr[fit_idx])
        prob = tree.predict_proba(X_tr.iloc[val_idx])[:, 1]
        fold_scores.append(roc_auc_score(y_tr[val_idx], prob,
                                         sample_weight=w_tr[val_idx]))
    cv_auc[d] = np.mean(fold_scores)
cv_auc = pd.Series(cv_auc, name="mean CV AUC")
best_depth = cv_auc.idxmax()
print(cv_auc.round(4).to_string())
print("Depth chosen by cross-validation:", best_depth)

## 29.5 Regularisation: lasso and ridge on a wage equation

In [ ]:
from sklearn.linear_model import Lasso, LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import r2_score

for c in ["sex", "marst", "gedu_lvl", "sec", "st", "aind_cws"]:
    sal[c] = sal[c].astype(int)
sal["edu_x_state"] = sal["gedu_lvl"] * 100 + sal["st"]     # one code per pair
sal["sex_x_ind"] = sal["sex"] * 100 + sal["aind_cws"]       # aind_cws: 2-digit NIC
sal["sec_x_state"] = sal["sec"] * 100 + sal["st"]
sal["age2"] = sal["age"] ** 2 / 100
NUM2 = ["age", "age2"]
CAT2 = ["gedu_lvl", "sex", "marst", "sec", "st", "aind_cws",
        "edu_x_state", "sex_x_ind", "sec_x_state"]

y_s = np.log(sal["ern_reg"]).to_numpy()
w_s = sal["weight"].to_numpy()
s_tr, s_te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=29)
                  .split(sal, y_s, groups=sal["mfsu"]))
enc = ColumnTransformer([("num", StandardScaler(), NUM2),
                         ("cat", OneHotEncoder(handle_unknown="ignore",
                                               min_frequency=5), CAT2)])
A_tr = enc.fit_transform(sal.iloc[s_tr])     # a sparse matrix: mostly zeros
A_te = enc.transform(sal.iloc[s_te])
names = enc.get_feature_names_out()
print(f"train {len(s_tr):,} workers, test {len(s_te):,}; "
      f"{A_tr.shape[1]} columns after making dummies")

In [ ]:
rng = np.random.default_rng(1)
few = rng.choice(len(s_tr), 1500, replace=False)
A_few, y_few = A_tr[few], y_s[s_tr][few]
w_few = w_s[s_tr][few] / w_s[s_tr][few].mean()


def r2(model, A, yy, ww):
    return r2_score(yy, model.predict(A), sample_weight=ww)


ols = LinearRegression().fit(A_few, y_few, sample_weight=w_few)
lasso = LassoCV(cv=5, random_state=0, max_iter=5000).fit(A_few, y_few,
                                                         sample_weight=w_few)
ridge = RidgeCV(alphas=np.logspace(-2, 4, 30)).fit(A_few, y_few,
                                                   sample_weight=w_few)
pd.DataFrame({
    "train R²": [r2(ols, A_few, y_few, w_few), r2(lasso, A_few, y_few, w_few),
                 r2(ridge, A_few, y_few, w_few)],
    "test R²": [r2(ols, A_te, y_s[s_te], w_s[s_te]),
                r2(lasso, A_te, y_s[s_te], w_s[s_te]),
                r2(ridge, A_te, y_s[s_te], w_s[s_te])],
    "non-zero coefficients": [(ols.coef_ != 0).sum(), (lasso.coef_ != 0).sum(),
                              (ridge.coef_ != 0).sum()]},
    index=["OLS", "Lasso (CV)", "Ridge (CV)"]).round(3)

In [ ]:
alphas = lasso.alphas_
cv_mse = lasso.mse_path_.mean(axis=1)
kept = [(Lasso(alpha=a, max_iter=5000).fit(A_few, y_few, sample_weight=w_few)
         .coef_ != 0).sum() for a in alphas[::4]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(alphas, cv_mse, color="#1f4e79")
ax1.axvline(lasso.alpha_, color="#c55a11", ls=":", lw=1.2)
ax1.set_xscale("log")
ax1.set_xlabel("Penalty λ (alpha), log scale")
ax1.set_ylabel("Cross-validated mean squared error")
ax1.set_title("Choosing the penalty", loc="left", fontsize=12)
ax2.plot(alphas[::4], kept, marker="o", ms=3, color="#1f4e79")
ax2.axvline(lasso.alpha_, color="#c55a11", ls=":", lw=1.2)
ax2.set_xscale("log")
ax2.set_xlabel("Penalty λ (alpha), log scale")
ax2.set_ylabel("Columns with a non-zero coefficient")
ax2.set_title("A bigger penalty keeps fewer variables", loc="left", fontsize=12)
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle("Lasso on a wage equation with 1,500 workers", x=0.01, ha="left",
             fontsize=13)
fig.tight_layout()
fig.text(0.01, -0.03, "Dotted line: penalty chosen by 5-fold CV. Log monthly "
         "earnings of regular salaried workers. Source: PLFS 2023-24 "
         "unit-level data (NSO, MoSPI), weighted.", fontsize=8, color="grey")
plt.show()

In [ ]:
w_all = w_s[s_tr] / w_s[s_tr].mean()
ols_all = LinearRegression().fit(A_tr, y_s[s_tr], sample_weight=w_all)
lasso_all = LassoCV(cv=5, random_state=0, max_iter=5000).fit(
    A_tr, y_s[s_tr], sample_weight=w_all)
print(f"All {len(s_tr):,} training workers:")
print(f"  OLS   test R² {r2(ols_all, A_te, y_s[s_te], w_s[s_te]):.3f}")
print(f"  Lasso test R² {r2(lasso_all, A_te, y_s[s_te], w_s[s_te]):.3f} "
      f"({(lasso_all.coef_ != 0).sum()} of {A_tr.shape[1]} columns kept)")

In [ ]:
chosen = []
for seed in range(5):
    idx = np.random.default_rng(100 + seed).choice(len(s_tr), 1500, replace=False)
    ww = w_s[s_tr][idx] / w_s[s_tr][idx].mean()
    m = LassoCV(cv=5, random_state=0, max_iter=5000).fit(A_tr[idx], y_s[s_tr][idx],
                                                         sample_weight=ww)
    chosen.append(set(names[m.coef_ != 0]))
in_all, in_any = set.intersection(*chosen), set.union(*chosen)
print("columns kept in each sample:", [len(c) for c in chosen])
print(f"kept in all five: {len(in_all)}; kept in at least one: {len(in_any)}")
print("always kept (first few):", sorted(in_all)[:8])

## 29.6 Random forests, and what "importance" means

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0)
best_tree.fit(X_tr, y_tr, sample_weight=w_tr)
results[f"Tree, depth {best_depth} (CV)"] = scores(best_tree.predict_proba(X_te)[:, 1])

start = time.time()
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=20,
                                max_features=0.5, max_samples=0.3,
                                random_state=0, n_jobs=1)
forest.fit(X_tr, y_tr, sample_weight=w_tr)
p_forest = forest.predict_proba(X_te)[:, 1]
results["Random forest"] = scores(p_forest)
print(f"forest grown in {time.time() - start:.0f} seconds")
table = pd.DataFrame(results).T
table.round(3)

In [ ]:
from sklearn.inspection import permutation_importance

pick = np.random.default_rng(0).choice(len(te), 15_000, replace=False)
imp = permutation_importance(forest, X_te.iloc[pick], y_te[pick],
                             scoring="roc_auc", n_repeats=3, random_state=0,
                             sample_weight=w_te[pick])
importance = pd.Series(imp.importances_mean, index=X.columns).sort_values()
LABELS = {"age": "Age", "hh_size": "Household size",
          "log_mpce": "Consumption per person (log)", "gedu_lvl": "Education",
          "marst": "Marital status", "rel": "Relation to head",
          "sec": "Rural/urban", "st": "State", "sg": "Social group",
          "relg": "Religion"}
importance.rename(LABELS).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(importance.rename(LABELS).index, importance.values, color="#1f4e79")
ax.set_xlabel("Fall in test AUC when the column is shuffled")
ax.set_title("What the forest leans on to predict women's participation",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Permutation importance, 15,000 test women, 3 "
         "shuffles each. Source: PLFS 2023-24 unit-level data (NSO, MoSPI), "
         "weighted.", fontsize=8, color="grey")
plt.show()

## 29.7 Text as data: collecting the RBI's statements

In [ ]:
TEXT_DIR = Path("data/raw/rbi_text")
LISTING = "https://www.rbi.org.in/Scripts/Annualpolicy.aspx"
PRESS = "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid="
web = requests.Session()
web.headers["User-Agent"] = ("python-for-indian-economics (teaching use; "
                             "textbook companion code)")


def polite_get(url, form=None):
    """GET (or POST a form), after a 1-second pause, decoded as UTF-8."""
    time.sleep(1)
    if form is None:
        reply = web.get(url, timeout=60)
    else:
        reply = web.post(url, data=form, timeout=60)
    reply.raise_for_status()
    reply.encoding = "utf-8"
    return reply.text


def listing_page(fy_end):
    """The Monetary Policy Statements page for the year ending March fy_end."""
    path = TEXT_DIR / f"listing_{fy_end}.html"
    if path.exists():
        return path.read_text(encoding="utf-8")
    first = polite_get(LISTING)
    form = dict(re.findall(r'<input type="hidden" name="([^"]+)" '
                           r'id="[^"]*" value="([^"]*)"', first))
    form.update({"hdnYear": str(fy_end), "UsrFontCntr$btn": ""})
    page = polite_get(LISTING, form)
    TEXT_DIR.mkdir(parents=True, exist_ok=True)
    path.write_text(page, encoding="utf-8")
    return page

In [ ]:
ANCHOR = re.compile(r"""<a [^>]*href=["']([^"']*)["'][^>]*>(.*?)</a>""",
                    re.S | re.I)
PHRASE = "Resolution of the Monetary Policy Committee"
INSIDE = re.compile(r"\{[^|{}]*prid=(\d+)\|[^}]*" + PHRASE)   # link text
AFTER = re.compile(PHRASE + r"[^{]*\{[^|{}]*prid=(\d+)\|")     # label, then link


def find_resolutions(page):
    rows, date = [], None
    for block in page.split('<td class="tableheader"><b>')[1:]:
        d = re.match(r"([A-Z][a-z]{2} \d{1,2}, \d{4})</b>", block)
        if d:                                   # a date header
            date = pd.to_datetime(d.group(1), format="%b %d, %Y")
            continue
        flat = ANCHOR.sub(lambda m: "{" + m.group(1) + "|"
                          + re.sub(r"<[^>]+>", " ", m.group(2)) + "}", block)
        flat = re.sub(r"<[^>]+>", " ", flat)
        for m in list(INSIDE.finditer(flat)) + list(AFTER.finditer(flat)):
            rows.append({"date": date, "prid": int(m.group(1))})
    return rows


def resolution_page(prid):
    """Plain text of one resolution, cached in data/raw/rbi_text/."""
    path = TEXT_DIR / f"mpc_resolution_{prid}.txt"
    if path.exists():
        return path.read_text(encoding="utf-8")
    page = polite_get(PRESS + str(prid))
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", page,
                  flags=re.S | re.I)
    text = re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", text)))
    path.write_text(text, encoding="utf-8")
    return text


rows = []
for fy_end in range(2017, 2028):          # 2016-17 ... 2026-27
    rows += find_resolutions(listing_page(fy_end))
docs = (pd.DataFrame(rows).drop_duplicates("prid").sort_values("date")
        .reset_index(drop=True))
docs["page"] = [resolution_page(n) for n in docs["prid"]]
print(f"{len(docs)} resolutions, {docs['date'].min():%d %b %Y} "
      f"to {docs['date'].max():%d %b %Y}")
docs.groupby(docs["date"].dt.year).size().to_frame("meetings").T

## 29.8 Cleaning and tokenising

In [ ]:
def body(page):
    start = re.search(r"Date : [A-Z][a-z]{2} \d{1,2}, \d{4}", page)
    end = re.search(r"Press Release\s?:\s?\d{4}-\d{4}/\d+", page)
    return page[start.end():end.start()].replace("�", "'").strip()


docs["text"] = docs["page"].map(body)
docs = docs.drop(columns="page")
docs["words"] = docs["text"].str.split().str.len()
print(docs["text"].iloc[-1][:400], "...")
early, late = docs["date"].dt.year <= 2019, docs["date"].dt.year >= 2023
print(f"median length: {docs.loc[early, 'words'].median():.0f} words in "
      f"2016-2019, {docs.loc[late, 'words'].median():.0f} in 2023-2026")
docs["words"].describe().round(0)

In [ ]:
def clean(text):
    text = text.lower().replace("-", " ")
    return re.sub(r"[^a-z ]", " ", text)


docs["clean"] = docs["text"].map(clean)
tokens = docs["clean"].str.split()
counts = Counter(w for doc in tokens for w in doc)
print(f"{sum(counts.values()):,} tokens, {len(counts):,} different words")
print(counts.most_common(12))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vec = CountVectorizer(stop_words="english", min_df=3)  # word in ≥ 3 documents
dtm = vec.fit_transform(docs["clean"])
vocab = vec.get_feature_names_out()
print(f"DTM: {dtm.shape[0]} documents × {dtm.shape[1]:,} words; "
      f"{1 - dtm.nnz / np.prod(dtm.shape):.0%} of cells are zero")
totals = pd.Series(np.asarray(dtm.sum(axis=0)).ravel(), index=vocab)
totals.sort_values(ascending=False).head(15).to_frame("count").T

## 29.9 TF-IDF and how much the language changes

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5)
T = tfidf.fit_transform(docs["clean"])
tf_vocab = tfidf.get_feature_names_out()


def distinctive(i, k=8):
    row = T[i].toarray().ravel()
    return ", ".join(tf_vocab[row.argsort()[::-1][:k]])


for when in ["2016-10-04", "2020-03-27", "2022-05-04", "2025-06-06"]:
    i = docs.index[docs["date"] == when][0]
    print(f"{docs.loc[i, 'date']:%b %Y}: {distinctive(i)}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(T)
docs["similarity"] = [np.nan] + [sim[i, i - 1] for i in range(1, len(docs))]
print(f"median similarity with the previous statement: "
      f"{docs['similarity'].median():.2f}")
breaks = docs.nsmallest(4, "similarity")
for i in breaks.index:
    print(f"{docs.loc[i, 'date']:%b %Y}  similarity {docs.loc[i, 'similarity']:.2f}"
          f"  distinctive: {distinctive(i, 6)}")

## 29.10 A hawkish/dovish dictionary

In [ ]:
HAWKISH = ["upside risk", "inflationary pressure", "price pressure",
           "elevated", "persistent", "persistence", "second round",
           "vigilant", "hardening", "firming", "overheating", "tightening",
           "withdrawal of accommodation"]
DOVISH = ["downside risk", "slowdown", "weak", "weakening", "subdued",
          "moderation", "moderated", "softening", "benign", "easing",
          "slack", "revive", "revival", "accommodative"]


def count_terms(text, terms):
    return {t: len(re.findall(r"\b" + t + r"s?\b", text)) for t in terms}


hawk = pd.DataFrame([count_terms(t, HAWKISH) for t in docs["clean"]])
dove = pd.DataFrame([count_terms(t, DOVISH) for t in docs["clean"]])
def ranked(table):
    s = table.sum().sort_values(ascending=False)
    return pd.DataFrame({"term": s.index, "uses": s.values.astype(object)})


pd.concat({"Hawkish": ranked(hawk), "Dovish": ranked(dove)},
          axis=1).astype(object).fillna("")

In [ ]:
def kwic(word, texts, k=6, n=5):
    out = []
    for date, text in texts:
        w = text.split()
        for i, tok in enumerate(w):
            if tok.startswith(word):
                out.append(f"{date:%b %Y}: ... {' '.join(w[max(0, i - k):i + k + 1])} ...")
    return out[::max(1, len(out) // n)][:n]


for line in kwic("elevated", zip(docs["date"], docs["clean"])):
    print(line)
after = Counter(re.findall(r"\belevated\s+(\w+)", " ".join(docs["clean"])))
print("\nword following 'elevated':", after.most_common(12))

In [ ]:
docs["H"], docs["D"] = hawk.sum(axis=1), dove.sum(axis=1)
docs["tone"] = (docs["H"] - docs["D"]) / (docs["H"] + docs["D"])
docs[["date", "words", "H", "D", "tone"]].tail(6).round({"tone": 2})

## 29.11 Does tone lead or follow policy?

In [ ]:
repo = bis("WS_CBPOL", "M.IN").set_index("date")["value"]
month = docs["date"].dt.to_period("M").dt.to_timestamp()
docs["repo"] = month.map(repo)
docs["change"] = ((docs["repo"] - month.map(repo.shift(1))) * 100).round()

VERB = re.compile(r"\b(reduc|increas|keep|maintain)\w*\b[^.]{0,120}?"
                  r"policy repo rate", re.I)
docs["verb"] = [VERB.search(t).group(1).lower() for t in docs["text"]]
print(f"BIS repo data end in {repo.index.max():%B %Y}; the latest resolution "
      f"({docs['date'].iloc[-1]:%d %b %Y}) says: {docs['verb'].iloc[-1]}...")
pd.crosstab(docs["verb"], np.sign(docs["change"]).map(
    {-1: "cut", 0: "no change", 1: "hike"}), margins=True)

In [ ]:
cpi = mospi_cpi("combined").set_index("date")["inflation"]
lag = np.where(docs["date"].dt.day >= 13, 1, 2)      # months back
known = [m - pd.DateOffset(months=int(k)) for m, k in zip(month, lag)]
docs["cpi_known"] = pd.Series(known).map(cpi).to_numpy()
print(f"CPI (base 2012) available to {cpi.index.max():%B %Y}; "
      f"meetings with a known CPI print: {docs['cpi_known'].notna().sum()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True,
                               gridspec_kw={"height_ratios": [1, 1.1]})
colors = np.where(docs["tone"] > 0, "#c55a11", "#1f4e79")
ax1.bar(docs["date"], docs["tone"], width=40, color=colors)
ax1.axhline(0, color="black", lw=0.8)
ax1.set_ylabel("Tone (−1 dovish, +1 hawkish)")
ax1.set_title("The tone of MPC resolutions, the repo rate and inflation",
              loc="left", fontsize=13)
ax2.step(repo.index, repo.values, where="post", color="#1f4e79",
         label="Repo rate, % (BIS)")
ax2.plot(cpi.index, cpi.values, color="#9db7d5", lw=1.2,
         label="CPI inflation, % y-o-y (MoSPI)")
ax2.axhspan(2, 6, color="grey", alpha=0.08)
ax2.set_xlim(pd.Timestamp("2016-07-01"), pd.Timestamp("2026-12-31"))
ax2.set_ylim(0, 9)
ax2.legend(loc="upper right", frameon=False, fontsize=9, ncol=2)
ax2.set_ylabel("Per cent")
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.text(0.01, -0.02, "Tone: dictionary index of each resolution (section "
         "29.10). Grey band: 2-6% tolerance band. Sources: RBI MPC resolutions "
         "(rbi.org.in); BIS policy rates; MoSPI CPI (base 2012).",
         fontsize=8, color="grey")
plt.show()

In [ ]:
leads = range(-3, 4)
lead_corr = pd.Series({k: docs["tone"].corr(docs["change"].shift(-k))
                       for k in leads})
n_pairs = {k: docs[["tone"]].join(docs["change"].shift(-k)).dropna().shape[0]
           for k in leads}
print(pd.DataFrame({"corr(tone_t, change_t+k)": lead_corr.round(2),
                    "pairs": n_pairs}).T.to_string())
print(f"corr(tone, CPI known at the meeting) = "
      f"{docs['tone'].corr(docs['cpi_known']):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(lead_corr.index, lead_corr.values,
       color=["#9db7d5" if k < 0 else "#1f4e79" if k == 0 else "#c55a11"
              for k in lead_corr.index])
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(list(leads))
ax.set_xticklabels([f"{k:+d}" if k else "0\n(same meeting)" for k in leads])
ax.set_xlabel("k: repo decision k meetings after (+) or before (−) the statement")
ax.set_ylabel("Correlation")
ax.set_title("Tone today is correlated with decisions before and after",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.11, "Correlation between the tone index at meeting t and "
         "the change in the repo rate (bps) at meeting t+k, Oct 2016 - Aug 2026."
         " Sources: RBI, BIS.", fontsize=8, color="grey")
plt.show()

In [ ]:
import statsmodels.formula.api as smf

reg = docs.assign(next_change=docs["change"].shift(-1)).dropna(
    subset=["next_change", "change", "cpi_known", "tone"])
base = smf.ols("next_change ~ change + cpi_known", data=reg).fit(
    cov_type="HAC", cov_kwds={"maxlags": 2})
with_tone = smf.ols("next_change ~ change + cpi_known + tone", data=reg).fit(
    cov_type="HAC", cov_kwds={"maxlags": 2})
print(f"meetings used: {int(with_tone.nobs)}  "
      f"({reg['date'].min():%b %Y} to {reg['date'].max():%b %Y})")
pd.DataFrame({"without tone": base.params, "with tone": with_tone.params,
              "with tone: p-value": with_tone.pvalues}).round(3).T.assign(
    adj_R2=[base.rsquared_adj, with_tone.rsquared_adj, np.nan]).round(3)

In [ ]:
follow = smf.ols("tone ~ cpi_known + prev_change",
                 data=docs.assign(prev_change=docs["change"].shift(1))).fit(
    cov_type="HAC", cov_kwds={"maxlags": 2})
print(f"meetings used: {int(follow.nobs)}")
pd.DataFrame({"coef": follow.params, "p-value": follow.pvalues}).round(3).T

## 29.12 Limitations, and a word on large language models

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading